In [2]:
import rasterio
import numpy as np
from pathlib import Path
from tqdm import tqdm

In [3]:
aggregated_dir = Path('aggregated_validation_dist_2024')

In [4]:
val_dist_s1_status_dir = Path('val_products_transformer_optimized-max10_processed_2026-02-05')
dist_s1_ts_dirs = list(val_dist_s1_status_dir.glob('*/'))

In [5]:
sample_ts_dir = dist_s1_ts_dirs[0]

In [6]:
status_files = sorted(list(sample_ts_dir.rglob('OPERA*_GEN-DIST-STATUS.tif')))
status_files[:3]

[PosixPath('val_products_transformer_optimized-max10_processed_2026-02-05/treelosswet__20KNG/OPERA_L3_DIST-ALERT-S1_T20KNG_20240111T094330Z_20260205T195303Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T20KNG_20240111T094330Z_20260205T195303Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('val_products_transformer_optimized-max10_processed_2026-02-05/treelosswet__20KNG/OPERA_L3_DIST-ALERT-S1_T20KNG_20240123T094330Z_20260205T193356Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T20KNG_20240123T094330Z_20260205T193356Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('val_products_transformer_optimized-max10_processed_2026-02-05/treelosswet__20KNG/OPERA_L3_DIST-ALERT-S1_T20KNG_20240204T094329Z_20260205T193410Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T20KNG_20240204T094329Z_20260205T193410Z_S1A_30_v0.1_GEN-DIST-STATUS.tif')]

In [7]:
def open_one_arr(path):
    with rasterio.open(path) as src:
        return src.read(1)

def aggregate_one(ts_dir: Path, glob_pattern: str = '*GEN-DIST-STATUS.tif') -> tuple:
    status_files = sorted(list(ts_dir.rglob(glob_pattern)))
    status_arrs = list(map(open_one_arr, status_files))
    status_arrs_stacked = np.stack(status_arrs, axis=0)

    confirmed_low_mask = (status_arrs_stacked == 3).any(axis=0)
    confirmed_high_mask = (status_arrs_stacked == 6).any(axis=0)

    status_agg = np.zeros(status_arrs_stacked.shape[1:], dtype=np.uint8)
    status_agg[confirmed_low_mask] = 3
    status_agg[confirmed_high_mask] = 6

    with rasterio.open(status_files[0]) as ds:
        p = ds.profile
        cmap = ds.colormap(1)

    return status_agg, p, cmap

def aggregate_and_serialize_one(ts_dir: Path, dst_dir: Path, glob_pattern: str = '*GEN-DIST-STATUS.tif', agg_token: str = 'dist_s1'):
    status_agg, p, cmap = aggregate_one(ts_dir)
    out_dir = dst_dir / ts_dir.name
    out_dir.mkdir(exist_ok=True, parents=True)
    out_path = out_dir / f'{agg_token}_agg_{ts_dir.name}.tif'
    with rasterio.open(out_path, 'w', **p) as dst:
        dst.write(status_agg, 1)
        dst.write_colormap(1, cmap)
    return out_path


In [25]:
_ = [aggregate_and_serialize_one(ts_dir, aggregated_dir) for ts_dir in tqdm(dist_s1_ts_dirs)]

NameError: name 'dist_s1_ts_dirs' is not defined

In [20]:
dist_hls_ts_dirs = sorted(list(Path('dist_hls/validation_sites').glob('*/')))
dist_hls_ts_dirs[:3], len(dist_hls_ts_dirs)

([PosixPath('dist_hls/validation_sites/builtnewalert__14RNU'),
  PosixPath('dist_hls/validation_sites/builtnewalert__16SBF'),
  PosixPath('dist_hls/validation_sites/builtnewalert__22KFA')],
 85)

In [21]:
_ = [aggregate_and_serialize_one(ts_dir, aggregated_dir, glob_pattern='*GEN-DIST-STATUS.tif', agg_token='dist_hls_gen') for ts_dir in tqdm(dist_hls_ts_dirs[:])]


100%|██████████| 85/85 [04:12<00:00,  2.97s/it]


In [22]:
_ = [aggregate_and_serialize_one(ts_dir, aggregated_dir, glob_pattern='*VEG-DIST-STATUS.tif', agg_token='dist_hls_veg') for ts_dir in tqdm(dist_hls_ts_dirs[:])]

100%|██████████| 85/85 [04:14<00:00,  2.99s/it]
